# Extract self-supervised frame embeddings

In [ ]:
from __future__ import annotations

import logging
import math
import sys
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
import torchaudio
import yaml
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, AutoModel

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print("CONFIG_PATH  =", CONFIG_PATH)

DEFAULT_CONFIG = {
    "paths": {
        "output_dir": "output",
        "log_dir": "output/logs",
        "manifest_csv": "output/00_manifest/manifest.csv",
    },
    "ssl": {
        "model_name": "facebook/wav2vec2-base",
        "target_sample_rate": 16000,
        "output_subdir": "01_ssl_frames",
        "save_dtype": "float16",
        "overwrite": False,
        "device": "auto",
        "normalize_peak": True,
        "min_duration_sec": 0.05,
        "save_frame_times": True,
        "max_clips": None,
    },
}

def deep_update(base: Dict[str, Any], updates: Dict[str, Any]) -> Dict[str, Any]:
    result = deepcopy(base)
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

CONFIG = deepcopy(DEFAULT_CONFIG)
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        loaded = yaml.safe_load(f) or {}
    CONFIG = deep_update(CONFIG, loaded)

def resolve_path(raw_path: str) -> Path:
    path = Path(raw_path)
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()

OUTPUT_DIR = resolve_path(CONFIG["paths"]["output_dir"])
LOG_DIR = resolve_path(CONFIG["paths"]["log_dir"])
MANIFEST_PATH = resolve_path(CONFIG["paths"]["manifest_csv"])
SSL_DIR = OUTPUT_DIR / CONFIG["ssl"]["output_subdir"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
SSL_DIR.mkdir(parents=True, exist_ok=True)

print("MANIFEST_PATH =", MANIFEST_PATH)
print("SSL_DIR       =", SSL_DIR)

In [ ]:
LOG_FILE = LOG_DIR / "01_extract_ssl_frames.log"

logger = logging.getLogger("stage4_ssl_frames")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.info("Logging initialized")
logger.info("Log file: %s", LOG_FILE)

In [ ]:
def choose_device(device_setting: str) -> torch.device:
    if device_setting == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")
    return torch.device(device_setting)

def sanitize_clip_id(value: str) -> str:
    text = str(value).strip()
    text = text.replace("/", "_").replace("\\", "_").replace(" ", "_")
    return text

def load_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Manifest file not found: {path}")
    df = pd.read_csv(path)
    required = {"clip_id", "audio_path"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest is missing required columns: {sorted(missing)}")
    df = df.copy()
    df["clip_id"] = df["clip_id"].astype(str).map(sanitize_clip_id)
    df["audio_path"] = df["audio_path"].astype(str)
    return df

def load_audio_mono(audio_path: Path, target_sr: int, normalize_peak: bool) -> tuple[np.ndarray, int, float]:
    waveform, sample_rate = torchaudio.load(str(audio_path))
    if waveform.ndim == 2 and waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    elif waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)

    if sample_rate != target_sr:
        waveform = torchaudio.functional.resample(waveform, sample_rate, target_sr)
        sample_rate = target_sr

    waveform = waveform.squeeze(0).to(torch.float32)

    if normalize_peak:
        peak = waveform.abs().max().item()
        if peak > 0:
            waveform = waveform / peak

    audio = waveform.cpu().numpy()
    duration_sec = len(audio) / float(sample_rate)
    return audio, sample_rate, duration_sec

def infer_frame_hop_sec(duration_sec: float, num_frames: int) -> float:
    if num_frames <= 0:
        return 0.0
    return float(duration_sec) / float(num_frames)

def build_frame_times(num_frames: int, frame_hop_sec: float) -> np.ndarray:
    if num_frames <= 0:
        return np.array([], dtype=np.float32)
    return (np.arange(num_frames, dtype=np.float32) * np.float32(frame_hop_sec)).astype(np.float32)

def existing_npz_index(npz_path: Path) -> Dict[str, Any]:
    with np.load(npz_path, allow_pickle=False) as data:
        embeddings = data["embeddings"]
        frame_times = data["frame_times_sec"] if "frame_times_sec" in data.files else None
        duration_sec = float(np.asarray(data["duration_sec"]).reshape(-1)[0]) if "duration_sec" in data.files else np.nan
        target_sr = int(np.asarray(data["target_sample_rate"]).reshape(-1)[0]) if "target_sample_rate" in data.files else np.nan
        frame_hop_sec = float(np.asarray(data["frame_hop_sec"]).reshape(-1)[0]) if "frame_hop_sec" in data.files else np.nan
        return {
            "num_frames": int(embeddings.shape[0]),
            "hidden_size": int(embeddings.shape[1]) if embeddings.ndim == 2 else np.nan,
            "saved_dtype": str(embeddings.dtype),
            "duration_sec": duration_sec,
            "target_sample_rate": target_sr,
            "frame_hop_sec": frame_hop_sec,
            "has_frame_times": frame_times is not None,
        }

def save_ssl_npz(
    output_path: Path,
    embeddings: np.ndarray,
    frame_times_sec: np.ndarray,
    duration_sec: float,
    original_sample_rate: int,
    target_sample_rate: int,
    frame_hop_sec: float,
    save_dtype: str,
) -> None:
    if save_dtype not in {"float16", "float32"}:
        raise ValueError(f"Unsupported save dtype: {save_dtype}")

    target_dtype = np.float16 if save_dtype == "float16" else np.float32
    embeddings_to_save = embeddings.astype(target_dtype, copy=False)

    np.savez_compressed(
        output_path,
        embeddings=embeddings_to_save,
        frame_times_sec=frame_times_sec.astype(np.float32, copy=False),
        duration_sec=np.array([duration_sec], dtype=np.float32),
        original_sample_rate=np.array([original_sample_rate], dtype=np.int32),
        target_sample_rate=np.array([target_sample_rate], dtype=np.int32),
        frame_hop_sec=np.array([frame_hop_sec], dtype=np.float32),
    )

In [ ]:
manifest_df = load_manifest(MANIFEST_PATH)

max_clips = CONFIG["ssl"].get("max_clips", None)
if max_clips is not None:
    manifest_df = manifest_df.head(int(max_clips)).copy()

missing_audio_mask = ~manifest_df["audio_path"].map(lambda p: Path(p).exists())
if missing_audio_mask.any():
    missing_count = int(missing_audio_mask.sum())
    display(manifest_df.loc[missing_audio_mask, ["clip_id", "audio_path"]].head(10))
    raise FileNotFoundError(f"{missing_count} manifest rows point to missing audio files.")

print("Rows to consider:", len(manifest_df))
display(manifest_df.head(5))

logger.info("Manifest loaded with %d rows", len(manifest_df))

In [ ]:
device = choose_device(CONFIG["ssl"]["device"])
model_name = CONFIG["ssl"]["model_name"]
target_sr = int(CONFIG["ssl"]["target_sample_rate"])

logger.info("Loading model: %s", model_name)
logger.info("Using device: %s", device)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

hidden_size = int(getattr(model.config, "hidden_size", 0))
print("Model       :", model_name)
print("Device      :", device)
print("Target SR   :", target_sr)
print("Hidden size :", hidden_size)

logger.info("Model ready | hidden_size=%s | target_sr=%s", hidden_size, target_sr)

In [ ]:
overwrite = bool(CONFIG["ssl"].get("overwrite", False))
normalize_peak = bool(CONFIG["ssl"].get("normalize_peak", True))
min_duration_sec = float(CONFIG["ssl"].get("min_duration_sec", 0.05))
save_dtype = str(CONFIG["ssl"].get("save_dtype", "float16")).lower()

frame_index_rows: List[Dict[str, Any]] = []
start_time = time.time()

for row in tqdm(manifest_df.itertuples(index=False), total=len(manifest_df), desc="Extracting SSL frames"):
    clip_id = sanitize_clip_id(row.clip_id)
    audio_path = Path(row.audio_path)
    output_path = SSL_DIR / f"{clip_id}.npz"

    if output_path.exists() and not overwrite:
        existing = existing_npz_index(output_path)
        frame_index_rows.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path),
            "frame_file": str(output_path),
            "duration_sec": existing["duration_sec"],
            "original_sample_rate": np.nan,
            "target_sample_rate": existing["target_sample_rate"],
            "num_frames": existing["num_frames"],
            "hidden_size": existing["hidden_size"],
            "frame_hop_sec": existing["frame_hop_sec"],
            "saved_dtype": existing["saved_dtype"],
            "status": "skipped_existing",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        continue

    if not audio_path.exists():
        logger.warning("Missing audio file for clip_id=%s | path=%s", clip_id, audio_path)
        frame_index_rows.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path),
            "frame_file": str(output_path),
            "duration_sec": np.nan,
            "original_sample_rate": np.nan,
            "target_sample_rate": target_sr,
            "num_frames": np.nan,
            "hidden_size": hidden_size,
            "frame_hop_sec": np.nan,
            "saved_dtype": save_dtype,
            "status": "missing_audio",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        continue

    try:
        waveform, original_sr = torchaudio.load(str(audio_path))
        if waveform.ndim == 2 and waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        elif waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        if original_sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, original_sr, target_sr)

        waveform = waveform.squeeze(0).to(torch.float32)

        if normalize_peak:
            peak = waveform.abs().max().item()
            if peak > 0:
                waveform = waveform / peak

        duration_sec = float(waveform.numel()) / float(target_sr)
        if duration_sec < min_duration_sec:
            logger.warning("Clip too short | clip_id=%s | duration_sec=%.5f", clip_id, duration_sec)
            frame_index_rows.append({
                "clip_id": clip_id,
                "audio_path": str(audio_path),
                "frame_file": str(output_path),
                "duration_sec": duration_sec,
                "original_sample_rate": original_sr,
                "target_sample_rate": target_sr,
                "num_frames": 0,
                "hidden_size": hidden_size,
                "frame_hop_sec": np.nan,
                "saved_dtype": save_dtype,
                "status": "too_short",
                "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            continue

        audio_np = waveform.cpu().numpy()

        inputs = feature_extractor(
            audio_np,
            sampling_rate=target_sr,
            return_tensors="pt",
            padding=False,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.squeeze(0).detach().cpu().numpy()

        num_frames = int(embeddings.shape[0])
        emb_hidden_size = int(embeddings.shape[1]) if embeddings.ndim == 2 else 0
        frame_hop_sec = infer_frame_hop_sec(duration_sec, num_frames)
        frame_times_sec = build_frame_times(num_frames, frame_hop_sec)

        save_ssl_npz(
            output_path=output_path,
            embeddings=embeddings,
            frame_times_sec=frame_times_sec,
            duration_sec=duration_sec,
            original_sample_rate=int(original_sr),
            target_sample_rate=target_sr,
            frame_hop_sec=frame_hop_sec,
            save_dtype=save_dtype,
        )

        frame_index_rows.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path),
            "frame_file": str(output_path),
            "duration_sec": duration_sec,
            "original_sample_rate": int(original_sr),
            "target_sample_rate": target_sr,
            "num_frames": num_frames,
            "hidden_size": emb_hidden_size,
            "frame_hop_sec": frame_hop_sec,
            "saved_dtype": save_dtype,
            "status": "processed",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        })

    except Exception as exc:
        logger.exception("Failed clip | clip_id=%s | path=%s | error=%s", clip_id, audio_path, exc)
        frame_index_rows.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path),
            "frame_file": str(output_path),
            "duration_sec": np.nan,
            "original_sample_rate": np.nan,
            "target_sample_rate": target_sr,
            "num_frames": np.nan,
            "hidden_size": hidden_size,
            "frame_hop_sec": np.nan,
            "saved_dtype": save_dtype,
            "status": f"error: {type(exc).__name__}",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        })

frame_index_df = pd.DataFrame(frame_index_rows)
FRAME_INDEX_PATH = SSL_DIR / "frame_index.csv"
frame_index_df.to_csv(FRAME_INDEX_PATH, index=False)

elapsed_sec = time.time() - start_time
status_counts = frame_index_df["status"].value_counts(dropna=False).to_dict()

logger.info("Frame index written to %s", FRAME_INDEX_PATH)
logger.info("Elapsed time: %.2f sec", elapsed_sec)
logger.info("Status counts: %s", status_counts)

print("FRAME_INDEX_PATH =", FRAME_INDEX_PATH)
print("Elapsed seconds  =", round(elapsed_sec, 2))
display(frame_index_df.head(10))

In [ ]:
if not frame_index_df.empty:
    processed_like = frame_index_df["status"].isin(["processed", "skipped_existing"])
    ok_df = frame_index_df.loc[processed_like].copy()

    print("Status counts:")
    display(frame_index_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

    if not ok_df.empty:
        summary = pd.DataFrame({
            "metric": [
                "clips_with_saved_frames",
                "mean_duration_sec",
                "median_duration_sec",
                "mean_num_frames",
                "median_num_frames",
                "mean_frame_hop_sec",
            ],
            "value": [
                len(ok_df),
                ok_df["duration_sec"].astype(float).mean(),
                ok_df["duration_sec"].astype(float).median(),
                ok_df["num_frames"].astype(float).mean(),
                ok_df["num_frames"].astype(float).median(),
                ok_df["frame_hop_sec"].astype(float).mean(),
            ],
        })
        display(summary)

        print("Preview of saved files:")
        display(ok_df[["clip_id", "frame_file", "duration_sec", "num_frames", "hidden_size", "saved_dtype", "status"]].head(10))
    else:
        print("No saved frame files found yet.")
else:
    print("frame_index_df is empty.")